In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, ListOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda, RunnableBranch

model = ChatOllama(model="gpt-oss:120b-cloud")

In [3]:
# Example 1 - Sequential Chain using pipe operator
prompt_1 = PromptTemplate(
    template="What is the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="Give comma separated list of 5 tourist attractions in {city}.",
    input_variables=["city"]
)
string_parser = StrOutputParser()
sequential_chain = prompt_1 | model | string_parser | prompt_2 | model | string_parser
response = sequential_chain.invoke({"country": "India"})
print("Sequential Chain Response:", response)


Sequential Chain Response: Red Fort, India Gate, Qutub Minar, Lotus Temple, Humayun's Tomb


In [ ]:
# Example 2 - Sequential Chain using RunnableSequence
prompt_1 = PromptTemplate(
    template="What is the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="Give comma separated list of 5 tourist attractions in {city}.",
    input_variables=["city"]
)
string_parser = StrOutputParser()
chain_1 = prompt_1 | model | string_parser
chain_2 = prompt_2 | model | string_parser
sequential_chain = RunnableSequence(chain_1, chain_2)
response = sequential_chain.invoke({"country": "India"})
print("Sequential Chain Response:", response)


Sequential Chain Response: Red Fort, India Gate, Qutub Minar, Lotus Temple, Humayun’s Tomb


In [8]:
# Example 3 - Parallel Chain using RunnableParallel
prompt_1 = PromptTemplate(
    template="What is the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="In which continent the city {city} is located?",
    input_variables=["city"]
)
string_parser = StrOutputParser()

parallel_chain = RunnableParallel({
    "capital_city": prompt_1 | model | string_parser,
    "continent":prompt_2 | model | string_parser
})
response = parallel_chain.invoke({"country": "India", "city": "Paris"})
print("Parallel Chain Response:", response)


Parallel Chain Response: {'capital_city': 'The capital city of India is **New\u202fDelhi**.', 'continent': 'Paris is located on the continent of **Europe**.'}


In [ ]:
# Example 3 - Parallel + Sequential Chain using RunnableParallel, RunnableSequence
prompt_1 = PromptTemplate(
    template="Which are the famous tourist places in the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="In which continent the country {country} is located?",
    input_variables=["country"]
)
string_parser = StrOutputParser()

parallel_chain = RunnableParallel({
    "tourist_places": prompt_1 | model | string_parser,
    "continent":prompt_2 | model | string_parser
})

prompt_3 = PromptTemplate(
    template="Write summarizing paragraph about the tourist places {tourist_places} and the continent {continent}",
    input_variables=["tourist_places", "continent"]
)

summary_chain = prompt_3 | model | string_parser
final_chain = parallel_chain | summary_chain
response = final_chain.invoke({"country": "India"})
print("Parallel + Sequential Chain Response:", response)


Parallel + Sequential Chain Response: New Delhi, the bustling capital of India (situated in South Asia), packs a remarkable mix of heritage, spirituality, greenery, and modern flair into a compact urban canvas. History lovers can wander the towering silhouettes of Red Fort, Qutub Minar, Humayun’s Tomb and the serene Jama Masjid, while the solemn India Gate and the striking National War Memorial offer poignant reminders of the nation’s past. Spiritual seekers find solace at the lotus‑shaped Baháʼí Temple, the grand Akshardham complex, the golden‑domed Bangla Sahib Gurudwara and the vibrant ISKCON shrine. Green retreats such as Lodi Gardens, the Garden of Five Senses and the expansive lawns around India Gate provide breathing room amid the city’s hustle, and cultural enthusiasts can dive into the nation’s artistic legacy at the National Museum, the National Gallery of Modern Art, and the Crafts Museum. For a taste of everyday Delhi, the chaotic lanes of Chandni Chowk, the colorful stalls